In [1]:
from data import api

In [ ]:
# get panel data
df = api.get_panel_data()
df

In [ ]:
from models.sarimax import sarimax

# 1. Prepare data with PCA on S0X
series_dict, meta_pca = sarimax.prepare_panel_for_sarimax(
    df_polars=df,                        # your Polars panel
    outcome="NUMBER_OF_HOMICIDIO",
    sector_cols=[f"S0{i}" for i in range(1, 9)],
    n_pcs=3,
)

# 2. Fit SARIMAX per dept
results_dict, summary_df = sarimax.fit_sarimax_panel(
    series_dict,
    order=(2, 0, 2),
    seasonal_order=(1, 0, 1, 12),
)
print(summary_df)

# 3. Global AIC/BIC
logL_total, AIC_total, BIC_total = sarimax.panel_aic_bic(summary_df)
print("Global SARIMAX AIC, BIC:", AIC_total, BIC_total)

# 4. Diagnostics for one department
dept = summary_df["DEPT_CODE"].iloc[0]
sarimax.sarimax_diagnostics(results_dict[dept], lags=24, title_prefix=f"Dept {dept}: ")


In [ ]:
for dept in summary_df["DEPT_CODE"]:
    sarimax.sarimax_diagnostics(results_dict[dept], lags=24, title_prefix=f"Dept {dept}: ")
    print(sarimax.sarimax_gof(results_dict[dept], lags=24, alpha=0.05, title_prefix="Dept 5:"))

In [ ]:
from models.sarimax.sarimax import build_sarimax_residuals_geodf, plot_sarimax_spatial_residuals

# Build GeoDataFrame with residual aggregates
gdf_resid = build_sarimax_residuals_geodf(
    results_dict,
    geojson_path="data/colombia_departments.geojson",
)

# Plot sum of absolute residuals
plot_sarimax_spatial_residuals(gdf_resid)


In [ ]:
dept_code = summary_df["DEPT_CODE"].iloc[5]
print(dept_code)
sarimax.sarimax_diagnostics(results_dict[dept_code], lags=24, title_prefix=f"Dept {dept_code}: ")
results_dict[dept_code].summary()

In [ ]:
from models.sarimax.sarimax import build_pc_sign_geodf, plot_pc_sign_maps

# After:
# series_dict, meta = prepare_panel_for_sarimax(df, outcome=..., n_pcs=3)
# results_dict, summary_df = fit_sarimax_panel(series_dict, ...)

geojson_path = "data/colombia_departments.geojson"

gdf_pc_sign = build_pc_sign_geodf(
    results_dict=results_dict,
    meta=meta_pca,
    geojson_path=geojson_path,
    alpha=0.05,
    dept_code_col_geo="dept_code_hecho",
)

plot_pc_sign_maps(
    gdf_pc_sign,
    pc_cols=meta_pca["pc_cols"],  # e.g. ["PC1","PC2","PC3"]
    title_prefix="SARIMAX PC coefficient sign (significant only)",
)



In [ ]:
from models.sarimax.sarimax import extract_significance_map, plot_pca_betas_spatially, plot_sign_map
import geopandas as gpd

# load your GeoJSON
gdf_dept = gpd.read_file("data/colombia_departments.geojson")
gdf_dept["dept_code_hecho"] = gdf_dept["dept_code_hecho"].astype(int)

# PCA betas
plot_pca_betas_spatially(results_dict, gdf_dept, meta_pca["pc_cols"])

# Seasonal terms
seasonal_names = ["ar.S.L12", "ma.S.L12"]
for name in seasonal_names:
    df_sign = extract_significance_map(results_dict, param_name=name)
    plot_sign_map(df_sign, gdf_dept, title=f"Significant sign of {name}")


# AR(1), MA(1)
ar_ma_names = ["ar.L1", "ma.L1"]  # extend as needed
for name in ar_ma_names:
    df_sign = extract_significance_map(results_dict, param_name=name)
    plot_sign_map(df_sign, gdf_dept, title=f"Significant sign of {name}")


# AR(1), MA(1)
ar_ma_names = ["ar.L2", "ma.L2"]  # extend as needed
for name in ar_ma_names:
    df_sign = extract_significance_map(results_dict, param_name=name)
    plot_sign_map(df_sign, gdf_dept, title=f"Significant sign of {name}")

# AR(1), MA(1)
ar_ma_names = ["POPULATION_std"]  # extend as needed
for name in ar_ma_names:
    df_sign = extract_significance_map(results_dict, param_name=name)
    plot_sign_map(df_sign, gdf_dept, title=f"Significant sign of {name}")


In [ ]:
from models.sarimax.sarimax import sarimax_in_sample_cv_panel, sarimax_rolling_cv_panel

metrics_in_sarimax = sarimax_in_sample_cv_panel(series_dict, results_dict, start_frac=0.5)
cv_out_sarimax = sarimax_rolling_cv_panel(df, "NUMBER_OF_HOMICIDIO")

cv_out_sarimax